# Шаг 12. Проверка и обработка дубликатов

In [2]:
import pandas as pd

# 1. Загружаем очищенный датасет (или используем текущий df из памяти)
df = pd.read_csv('df_consolidated_clean.csv')

print("=== 1. Проверка полных дубликатов строк ===")
# Считаем количество строк, которые полностью повторяются
full_duplicates_count = df.duplicated().sum()
print(f"Найдено полных дубликатов строк: {full_duplicates_count}")

if full_duplicates_count > 0:
    print("\nПримеры дубликатов:")
    # Показываем строки, которые являются дубликатами (keep=False покажет все копии)
    display(df[df.duplicated(keep=False)].head(10))
    
    # Удаляем полные дубликаты и сбрасываем индекс
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"\nПосле удаления дубликатов в датасете осталось строк: {len(df)}")
else:
    print("Полных дубликатов строк не найдено. Структура данных корректна!")

print("\n=== 2. Проверка на смысловые дубликаты (категориальные поля) ===")
# Проверяем, нет ли разных написаний одного и того же производителя
print("Топ-15 уникальных производителей:")
print(df['manufacturer'].value_counts().head(15))

print("\nТоп-20 уникальных национальностей:")
print(df['nationality'].value_counts().head(20))

=== 1. Проверка полных дубликатов строк ===
Найдено полных дубликатов строк: 0
Полных дубликатов строк не найдено. Структура данных корректна!

=== 2. Проверка на смысловые дубликаты (категориальные поля) ===
Топ-15 уникальных производителей:
manufacturer
Strelets         640
HaT              512
Zvezda           241
RedBox           237
Mars             214
Caesar           174
Linear-A         158
Italeri          158
Orion            134
Preiser          122
Revell           116
Airfix            83
Esci              81
LW                77
Waterloo 1815     71
Name: count, dtype: int64

Топ-20 уникальных национальностей:
nationality
German        458
Не указано    364
British       346
French        301
Russian       263
USA           262
Italian       251
Roman         140
Soviet        110
Spanish        79
Japanese       68
Turkish        62
Austrian       59
Prussian       56
Polish         55
Greek          46
English        42
Ottoman        38
Ukrainian      33
Swedish      

## Результат проверки
Полных дубликатов не выявлено.

Выявлены смысловые дубликаты национальностей.

## Решение
Будет произведена группировка.

In [12]:
import pandas as pd
df = pd.read_csv('df_consolidated_clean.csv')

# 1. Определяем актуальное имя столбца 
nat_col = 'nationality' if 'nationality' in df.columns else 'Nationality'

# 2. Словарь для объединения исторически связанных категорий
# Ключи — это старые значения, значения — это новые объединенные категории
grouping_map = {
    'Soviet': 'Russian',
    'Prussian': 'German',
    'Roman': 'Italian',
    'Ottoman': 'Turkish',
    'English': 'British',
    'Habsburg Imperial' : 'Austrian',
    'Muscovite': 'Russian',
    'Cossac': 'Ukrainian',
    'Judean': 'Israeli',
    'Byzantine': 'Greek',
    'Iranian': 'Persian'
}

# 3. Применяем замену
# Метод replace заменит только точные совпадения из ключей словаря
df[nat_col] = df[nat_col].replace(grouping_map)

# 4. Дополнительная проверка: убедимся, что старые названия исчезли
print("\n=== Проверка исчезновения старых категорий ===")
old_categories = ['Soviet', 'Prussian', 'Roman', 'Ottoman', 'English', 'Habsburg Imperial', 'Muscovite', 'Cossac', 'Judean', 'Byzantine', 'Iranian']
for cat in old_categories:
    count = (df[nat_col] == cat).sum()
    print(f"Записей с названием '{cat}': {count}")

# 6. Сохраняем изменения в CSV
df.to_csv('df_consolidated.csv', index=False, encoding='utf-8')
print("\n✅ Изменения сохранены в 'df_consolidated.csv'")


=== Проверка исчезновения старых категорий ===
Записей с названием 'Soviet': 0
Записей с названием 'Prussian': 0
Записей с названием 'Roman': 0
Записей с названием 'Ottoman': 0
Записей с названием 'English': 0
Записей с названием 'Habsburg Imperial': 0
Записей с названием 'Muscovite': 0
Записей с названием 'Cossac': 0
Записей с названием 'Judean': 0
Записей с названием 'Byzantine': 0
Записей с названием 'Iranian': 0

✅ Изменения сохранены в 'df_consolidated.csv'


Группировка могла привести к появлению полных дубликатов строк.
## Решение
Будет проведена повторная проверка на дубликаты.

In [13]:
# 1. Удаление полных дубликатов
# keep='first' оставляет первую встреченную уникальную комбинацию всех столбцов
df = df.drop_duplicates().reset_index(drop=True)

# 2. Проверка на полные дубликаты ПОСЛЕ очистки
print("\n=== Количество полных дубликатов ПОСЛЕ очистки ===")
duplicates_after = df.duplicated().sum()
print(f"Найдено дубликатов: {duplicates_after}")
print(f"Итоговое количество строк в датасете: {len(df)}")

# 3. Финальная проверка топ-национальностей (чтобы убедиться в чистоте данных)
print("\n=== Топ-15 национальностей после группировки и удаления дубликатов ===")
print(df['nationality'].value_counts().head(15))

# 4. Сохранение окончательно очищенного датасета
df.to_csv('df_consolidated_clean.csv', index=False, encoding='utf-8')
print("\n✅ Итоговый очищенный датасет сохранен как 'df_consolidated_clean.csv'")


=== Количество полных дубликатов ПОСЛЕ очистки ===
Найдено дубликатов: 0
Итоговое количество строк в датасете: 3506

=== Топ-15 национальностей после группировки и удаления дубликатов ===
nationality
German        458
Не указано    364
British       348
French        301
Russian       263
USA           262
Italian       257
Spanish        79
Japanese       68
Turkish        62
Austrian       62
Greek          59
Polish         55
Ukrainian      33
Swedish        31
Name: count, dtype: int64

✅ Итоговый очищенный датасет сохранен как 'df_consolidated_clean.csv'


## Результат шага
- Выполнена смысловая группировка исторически связанных национальностей.
- Выявлены и удалены полные дубликаты строк, возникшие в результате группировки.
- Датасет приведен к чистому, непротиворечивому виду и сохранен для дальнейшего анализа.